# AI-Based Skin Disease Detection with Treatment Cost Estimation
## Phase 4: Vision Transformer (ViT-B/16) Disease Classification Training

This ready-to-run Google Colab notebook trains and evaluates the **Vision Transformer (ViT-B/16)** model across the 7 dermatological disease categories using transfer learning and discriminative learning rates.

- **Architecture:** `vit_base_patch16_224` (timm, ImageNet-21k fine-tuned on ImageNet-1k)
- **Categories:** MEL (0), NV (1), BCC (2), AKIEC (3), BKL (4), DF (5), VASC (6)
- **Loss Function:** Weighted Cross-Entropy Loss (handles class imbalance)
- **Dataset:** 1,233 Train / 267 Validation / 257 Held-Out Test images
- **Optimization:** AdamW (Head lr=1e-4, Backbone lr=1e-5), Warmup + Cosine Annealing, Early Stopping (Patience=5)
- **Hardware:** GPU Acceleration (T4 / V100 / A100)

### Step 1: Verify GPU Environment

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

### Step 2: Install Necessary Dependencies

In [ ]:
%pip install -q timm opencv-python matplotlib pandas scikit-learn

### Step 3: Extract the Classification Dataset Package
Upload `colab_vit_package.zip` into the Colab files pane, then run this cell.

In [ ]:
import os, sys, zipfile
from pathlib import Path

if os.path.exists('colab_vit_package.zip') and not os.path.exists('models/vit.py'):
    print("Unzipping colab_vit_package.zip...")
    with zipfile.ZipFile('colab_vit_package.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete.")

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

assert os.path.exists('models/vit.py'), "models/vit.py not found!"
assert os.path.exists('data/manifests/classification_train.csv'), "classification_train.csv not found!"
print("Project structure and manifests verified.")

### Step 4: Verify Dataset and DataLoaders

In [ ]:
from training.dataset import get_classification_dataloaders, compute_class_weights
from models.vit import DISEASE_CLASSES, CLASS_DISPLAY_NAMES

loaders = get_classification_dataloaders(batch_size=16, num_workers=2)
train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

print(f"Train samples: {len(train_loader.dataset)} (78 batches of 16)")
print(f"Validation samples: {len(val_loader.dataset)} (17 batches)")
print(f"Held-Out Test samples: {len(test_loader.dataset)} (17 batches)")

weights = compute_class_weights('data/manifests/classification_train.csv')
print("\nComputed Class Weights for Cross-Entropy Loss:")
for c_idx, c_name in enumerate(DISEASE_CLASSES):
    print(f"  {c_name.upper()} ({CLASS_DISPLAY_NAMES[c_name]}): {weights[c_idx]:.4f}")

for imgs, labels, ids in train_loader:
    print(f"\nSample batch image shape: {imgs.shape} (Expected: [16, 3, 224, 224])")
    print(f"Sample batch labels:      {labels.tolist()[:8]}")
    break

### Step 5: Train the Vision Transformer (ViT-B/16) Model
Fine-tunes pretrained ViT-B/16 with discriminative learning rates, weighted cross-entropy, and early stopping.

In [ ]:
from training.train_vit import run_training

history = run_training(
    epochs=20,
    batch_size=16,
    head_lr=1e-4,
    backbone_lr=1e-5,
    patience=5,
    freeze_backbone=True,
    unfreeze_blocks=2,
    smoke_test=False
)

### Step 6: Visualize Training Curves

In [ ]:
from IPython.display import Image, display
import os

print("Training vs Validation Loss Curve:")
if os.path.exists('training/results/vit_plots/loss_curve.png'):
    display(Image('training/results/vit_plots/loss_curve.png'))

print("Training vs Validation Accuracy Curve:")
if os.path.exists('training/results/vit_plots/acc_curve.png'):
    display(Image('training/results/vit_plots/acc_curve.png'))

print("Validation Macro F1-Score Curve:")
if os.path.exists('training/results/vit_plots/f1_curve.png'):
    display(Image('training/results/vit_plots/f1_curve.png'))

### Step 7: Display Confusion Matrix and Genuine Test Predictions

In [ ]:
import glob

print("Confusion Matrix Heatmap:")
if os.path.exists('training/results/vit_plots/confusion_matrix.png'):
    display(Image('training/results/vit_plots/confusion_matrix.png'))

print("\nMulticlass One-vs-Rest ROC Curves:")
if os.path.exists('training/results/vit_plots/roc_curves.png'):
    display(Image('training/results/vit_plots/roc_curves.png'))

print("\nSample Test Prediction Cards (Correct & Misclassified):")
sample_files = sorted(glob.glob('training/results/vit_samples/*.png'))
for f in sample_files[:4]:
    display(Image(f))

### Step 8: Display Final Held-Out Test Evaluation Metrics

In [ ]:
import json

with open('training/results/vit_metrics.json') as f:
    metrics = json.load(f)

print("=" * 55)
print("FINAL HELD-OUT TEST EVALUATION METRICS (ViT-B/16)")
print("=" * 55)
for k, v in metrics.items():
    if k == 'per_class':
        continue
    if isinstance(v, float):
        print(f"{k.replace('_', ' ').title():<25}: {v:.4f}")
    else:
        print(f"{k.replace('_', ' ').title():<25}: {v}")

print("\n" + "-" * 55)
print("PER-CLASS PERFORMANCE BREAKDOWN")
print("-" * 55)
for c_code, c_data in metrics.get('per_class', {}).items():
    name = c_data['display_name']
    p = c_data['precision']
    r = c_data['recall']
    f1 = c_data['f1_score']
    supp = c_data['support']
    print(f"{c_code.upper()} ({name:<30}): P={p:.3f} | R={r:.3f} | F1={f1:.3f} | N={supp}")
print("=" * 55)

### Step 9: Download Trained Weights & Evaluation Artifacts

In [ ]:
import zipfile
from google.colab import files

# Create results archive for clean download
with zipfile.ZipFile('vit_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('training/results/vit_metrics.json', 'vit_metrics.json')
    z.write('training/results/vit_training_history.csv', 'vit_training_history.csv')
    for f in glob.glob('training/results/vit_plots/*.png'):
        z.write(f, os.path.relpath(f, 'training/results'))
    for f in glob.glob('training/results/vit_samples/*.png'):
        z.write(f, os.path.relpath(f, 'training/results'))

print("Downloading vit_results.zip and vit_model.pth...")
files.download('vit_results.zip')
if os.path.exists('models/vit_model.pth'):
    files.download('models/vit_model.pth')
    files.download('training/results/vit_metrics.json')